In [0]:
import yaml
from datetime import datetime

# ── Create Silver schema ───────────────────────────────────────────────────
spark.sql("CREATE SCHEMA IF NOT EXISTS sales.silver")
print("✓ Schema sales.silver ready\n")

# ── Load SQL transforms from YAML ──────────────────────────────────────────
with open("/Volumes/sales/mapping/mappings/bronze_silver_transforms.yaml") as f:
    transforms = yaml.safe_load(f)["tables"]

# ── Execute each Bronze → Silver transform ─────────────────────────────────
print(f"Bronze -> Silver  |  {len(transforms)} tables\n")
print(f"  {'Table':<50} {'Rows':>10}  {'Time':>6}")
print(f"  {'-' * 70}")

for target, sql in transforms.items():
    t0 = datetime.now()
    try:
        spark.sql(sql)
        cnt     = spark.table(target).count()
        elapsed = round((datetime.now() - t0).total_seconds(), 1)
        print(f"  {target:<50} {cnt:>10,}  {elapsed:>5.1f}s")
    except Exception as e:
        print(f"  {target:<50} {'ERROR':>10}  {str(e)[:60]}")

print(f"\n  Done — {len(transforms)} tables written to sales.silver")

✓ Schema sales.silver ready

Bronze -> Silver  |  10 tables

  Table                                                    Rows    Time
  ----------------------------------------------------------------------
  sales.silver.category_lookup                               71    5.4s
  sales.silver.customers                                 99,441    4.7s
  sales.silver.geolocation                               19,015    4.6s
  sales.silver.order_items                              112,650    4.1s
  sales.silver.order_payments                           103,886    4.0s
  sales.silver.order_reviews                             99,224    3.9s
  sales.silver.orders                                    99,441    4.2s
  sales.silver.products                                  32,951    3.8s
  sales.silver.reviews_bad_rows                               0    3.6s
  sales.silver.sellers                                    3,095    3.9s

  Done — 10 tables written to sales.silver
